# Лабораторная работа №6. Логистическая регрессия## Датасет: Titanic Dataset (пассажиры Титаника)

## 1. Импорт библиотек

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerimport warningswarnings.filterwarnings('ignore')sns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (10, 6)plt.rcParams['font.size'] = 12print('Все библиотеки загружены')

## 2. Загрузка датасета

In [ ]:
df = sns.load_dataset('titanic')print(f'Размер датасета: {df.shape[0]} строк × {df.shape[1]} столбцов')print(f'Столбцы: {df.columns.tolist()}')df.head(10)

## 3. Анализ пропущенных значений

In [ ]:
missing = df.isnull().sum()missing_pct = (missing / len(df) * 100).round(2)missing_df = pd.DataFrame({'Пропущено': missing, 'Процент': missing_pct})missing_df[missing_df['Пропущено'] > 0].sort_values('Пропущено', ascending=False)

## 4. Предварительная обработка данных

In [ ]:
features = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()target = df['survived'].copy()print('Распределение целевой переменной:')print(f'  Выжило (1):  {target.sum()} чел. ({target.mean()*100:.1f}%)')print(f'  Погибло (0): {(1-target).sum()} чел. ({(1-target).mean()*100:.1f}%)')features['sex'] = (features['sex'] == 'female').astype(int)features = pd.get_dummies(features, columns=['embarked'], drop_first=True, dtype=int)features = pd.get_dummies(features, columns=['pclass'], drop_first=True, dtype=int, prefix='pclass')features['age'] = features['age'].fillna(features['age'].median())features['fare'] = features['fare'].fillna(features['fare'].median())print(f'\nПризнаки после обработки ({features.shape[1]} шт.):')print(features.columns.tolist())print(f'Осталось пропусков: {features.isnull().sum().sum()}')features.head()

## 5. Статистика по датасету

In [ ]:
stats = features.describe().Tstats['median'] = features.median()stats = stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]stats.round(3)

## 6. Визуализация распределения признаков

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))axes = axes.flatten()feat_list = ['age', 'fare', 'sibsp', 'parch', 'sex', 'embarked_S']titles_ru = ['Возраст', 'Стоимость билета (Fare)', 'SibSp (родственники)',             'Parch (родители / дети)', 'Пол (0 = М, 1 = Ж)', 'Порт Southampton (0/1)']colors = ['#3498DB', '#2ECC71', '#E67E22', '#9B59B6', '#E74C3C', '#1ABC9C']for i, (feat, title, clr) in enumerate(zip(feat_list, titles_ru, colors)):    if feat in ['sex', 'embarked_S']:        cnt = features[feat].value_counts()        axes[i].bar(['0', '1'], [cnt.get(0, 0), cnt.get(1, 0)], color=[clr, '#95A5A6'], edgecolor='black')        axes[i].set_title(title, fontsize=12, fontweight='bold')        axes[i].set_ylabel('Количество')    else:        axes[i].hist(features[feat].dropna(), bins=25, color=clr, edgecolor='black', alpha=0.85)        axes[i].axvline(features[feat].mean(), color='red', ls='--', lw=2, label=f'Среднее: {features[feat].mean():.2f}')        axes[i].axvline(features[feat].median(), color='green', ls='--', lw=2, label=f'Медиана: {features[feat].median():.2f}')        axes[i].set_title(title, fontsize=12, fontweight='bold')        axes[i].legend(fontsize=8)        axes[i].set_ylabel('Частота')axes[-1].set_visible(False)plt.suptitle('Распределение признаков датасета Titanic', fontsize=15, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

## 7. Нормализация и разделение данных

In [ ]:
scaler = StandardScaler()X_scaled = scaler.fit_transform(features)X_scaled = pd.DataFrame(X_scaled, columns=features.columns)X = X_scaled.valuesy = target.values.reshape(-1, 1)X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)print(f'Обучающая выборка: {X_train.shape[0]} образцов ({X_train.shape[0]/len(y)*100:.0f}%)')print(f'Тестовая выборка:   {X_test.shape[0]} образцов ({X_test.shape[0]/len(y)*100:.0f}%)')print(f'Доля выживших — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}')

## 8. Реализация логистической регрессии (с нуля)

### 8.1 Класс с градиентным спуском

In [ ]:
class LogisticRegressionGD:    """Логистическая регрессия с градиентным спуском."""    def __init__(self, learning_rate=0.01, n_iterations=1000, verbose=False):        self.lr = learning_rate        self.n_iter = n_iterations        self.verbose = verbose        self.weights = None        self.bias = None        self.loss_history = []    @staticmethod    def _sigmoid(z):        z = np.clip(z, -500, 500)        return 1 / (1 + np.exp(-z))    def fit(self, X, y):        n_samples, n_features = X.shape        self.weights = np.zeros((n_features, 1))        self.bias = 0.0        self.loss_history = []        for i in range(self.n_iter):            linear = np.dot(X, self.weights) + self.bias            y_pred = self._sigmoid(linear)            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))            db = (1 / n_samples) * np.sum(y_pred - y)            self.weights -= self.lr * dw            self.bias -= self.lr * db            loss = self._log_loss(y, y_pred)            self.loss_history.append(loss)            if self.verbose and i % max(1, self.n_iter // 10) == 0:                print(f'  Итерация {i:5d}: loss = {loss:.6f}')        return self    def predict_proba(self, X):        linear = np.dot(X, self.weights) + self.bias        return self._sigmoid(linear)    def predict(self, X, threshold=0.5):        return (self.predict_proba(X) >= threshold).astype(int)    @staticmethod    def _log_loss(y_true, y_pred):        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))print('LogisticRegressionGD — готово')

### 8.2 Класс с методом Ньютона (Newton-Raphson)

In [ ]:
class LogisticRegressionNewton:    """Логистическая регрессия с методом Ньютона."""    def __init__(self, n_iterations=15, tol=1e-6, verbose=False):        self.n_iter = n_iterations        self.tol = tol        self.verbose = verbose        self.weights = None        self.bias = None        self.loss_history = []    @staticmethod    def _sigmoid(z):        z = np.clip(z, -500, 500)        return 1 / (1 + np.exp(-z))    def fit(self, X, y):        n_samples, n_features = X.shape        self.weights = np.zeros((n_features, 1))        self.bias = 0.0        self.loss_history = []        for i in range(self.n_iter):            linear = np.dot(X, self.weights) + self.bias            y_pred = self._sigmoid(linear)            loss = self._log_loss(y, y_pred)            self.loss_history.append(loss)            grad_w = (1 / n_samples) * np.dot(X.T, (y_pred - y))            grad_b = (1 / n_samples) * np.sum(y_pred - y)            grad = np.vstack([grad_w, [[grad_b]]])            S = np.diag((y_pred * (1 - y_pred)).flatten())            X_aug = np.hstack([X, np.ones((n_samples, 1))])            H = (1 / n_samples) * np.dot(np.dot(X_aug.T, S), X_aug)            H_reg = H + 1e-5 * np.eye(H.shape[0])            try:                delta = np.linalg.solve(H_reg, grad)            except np.linalg.LinAlgError:                delta = np.linalg.lstsq(H_reg, grad, rcond=None)[0]            self.weights -= delta[:-1].reshape(-1, 1)            self.bias -= delta[-1, 0]            if self.verbose:                print(f'  Итерация {i:3d}: loss = {loss:.6f}, |grad| = {np.linalg.norm(grad):.2e}')            if np.linalg.norm(grad) < self.tol:                if self.verbose:                    print(f'  Сходимость достигнута на итерации {i}')                break        return self    def predict_proba(self, X):        linear = np.dot(X, self.weights) + self.bias        return self._sigmoid(linear)    def predict(self, X, threshold=0.5):        return (self.predict_proba(X) >= threshold).astype(int)    @staticmethod    def _log_loss(y_true, y_pred):        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))print('LogisticRegressionNewton — готово')

## 9. Метрики (реализация с нуля)

In [ ]:
def compute_metrics(y_true, y_pred):    tp = int(np.sum((y_true == 1) & (y_pred == 1)))    tn = int(np.sum((y_true == 0) & (y_pred == 0)))    fp = int(np.sum((y_true == 0) & (y_pred == 1)))    fn = int(np.sum((y_true == 1) & (y_pred == 0)))    accuracy = (tp + tn) / len(y_true)    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0    return {        'Accuracy': round(accuracy, 4),        'Precision': round(precision, 4),        'Recall': round(recall, 4),        'F1-Score': round(f1, 4),        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn    }print('Функция compute_metrics — готова')

## 10. Исследование гиперпараметров

### 10.1 Влияние learning rate (градиентный спуск)

In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]gd_lr_results = []for lr in learning_rates:    model = LogisticRegressionGD(learning_rate=lr, n_iterations=5000)    model.fit(X_train, y_train)    y_pred = model.predict(X_test)    m = compute_metrics(y_test, y_pred)    m['lr'] = lr    gd_lr_results.append(m)    print(f'lr={lr:<8}  Acc={m["Accuracy"]:.4f}  Prec={m["Precision"]:.4f}  '          f'Rec={m["Recall"]:.4f}  F1={m["F1-Score"]:.4f}')df_lr = pd.DataFrame(gd_lr_results)[['lr', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]df_lr

### 10.2 Влияние количества итераций (градиентный спуск)

In [ ]:
n_iters_list = [100, 500, 1000, 5000, 10000]gd_iter_results = []for n_iter in n_iters_list:    model = LogisticRegressionGD(learning_rate=0.1, n_iterations=n_iter)    model.fit(X_train, y_train)    y_pred = model.predict(X_test)    m = compute_metrics(y_test, y_pred)    m['n_iter'] = n_iter    gd_iter_results.append(m)    print(f'n_iter={n_iter:<6}  Acc={m["Accuracy"]:.4f}  Prec={m["Precision"]:.4f}  '          f'Rec={m["Recall"]:.4f}  F1={m["F1-Score"]:.4f}')df_iter = pd.DataFrame(gd_iter_results)[['n_iter', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]df_iter

### 10.3 Метод Ньютона

In [ ]:
newton_results = []for n_iter in [3, 5, 10, 15, 20, 30, 50]:    model = LogisticRegressionNewton(n_iterations=n_iter)    model.fit(X_train, y_train)    y_pred = model.predict(X_test)    m = compute_metrics(y_test, y_pred)    m['n_iter'] = n_iter    newton_results.append(m)    print(f'n_iter={n_iter:<3}  Acc={m["Accuracy"]:.4f}  Prec={m["Precision"]:.4f}  '          f'Rec={m["Recall"]:.4f}  F1={m["F1-Score"]:.4f}')df_newton = pd.DataFrame(newton_results)[['n_iter', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]df_newton

## 11. Визуализация результатов исследования

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))# Learning rateax = axes[0, 0]for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:    ax.plot([r['lr'] for r in gd_lr_results], [r[m] for r in gd_lr_results],            marker='o', lw=2, label=m, markersize=8)ax.set_xscale('log')ax.set_xlabel('Learning Rate (log scale)'); ax.set_ylabel('Значение метрики')ax.set_title('Влияние learning rate (n_iter=5000)', fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0.5, 1.0)# Количество итераций (GD)ax = axes[0, 1]for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:    ax.plot([r['n_iter'] for r in gd_iter_results], [r[m] for r in gd_iter_results],            marker='s', lw=2, label=m, markersize=8)ax.set_xscale('log')ax.set_xlabel('Количество итераций (log scale)'); ax.set_ylabel('Значение метрики')ax.set_title('Влияние количества итераций (lr=0.1)', fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0.5, 1.0)# Метод Ньютонаax = axes[1, 0]for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:    ax.plot([r['n_iter'] for r in newton_results], [r[m] for r in newton_results],            marker='D', lw=2, label=m, markersize=8)ax.set_xlabel('Количество итераций'); ax.set_ylabel('Значение метрики')ax.set_title('Метод Ньютона: влияние количества итераций', fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0.5, 1.0)# Сравнение сходимостиax = axes[1, 1]gd = LogisticRegressionGD(learning_rate=0.1, n_iterations=100); gd.fit(X_train, y_train)nt = LogisticRegressionNewton(n_iterations=100); nt.fit(X_train, y_train)ax.plot(gd.loss_history, label='Градиентный спуск (lr=0.1)', lw=2, color='#3498DB')ax.plot(nt.loss_history, label='Метод Ньютона', lw=2, color='#E74C3C')ax.set_xlabel('Итерация'); ax.set_ylabel('Log Loss')ax.set_title('Сравнение сходимости: GD vs Newton', fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3)plt.suptitle('Исследование гиперпараметров логистической регрессии', fontsize=15, fontweight='bold')plt.tight_layout()plt.show()

## 12. Тепловая карта: Accuracy(GD) для всех комбинаций (lr × n_iter)

In [ ]:
learning_rates_all = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]n_iters_all = [100, 500, 1000, 5000, 10000]heatmap_acc = np.zeros((len(learning_rates_all), len(n_iters_all)))for i, lr in enumerate(learning_rates_all):    for j, n_iter in enumerate(n_iters_all):        model = LogisticRegressionGD(learning_rate=lr, n_iterations=n_iter)        model.fit(X_train, y_train)        y_pred = model.predict(X_test)        heatmap_acc[i, j] = compute_metrics(y_test, y_pred)['Accuracy']fig, ax = plt.subplots(figsize=(10, 6))im = ax.imshow(heatmap_acc, cmap='YlOrRd', aspect='auto', vmin=0.60, vmax=0.85)ax.set_xticks(range(len(n_iters_all))); ax.set_xticklabels(n_iters_all)ax.set_yticks(range(len(learning_rates_all))); ax.set_yticklabels(learning_rates_all)ax.set_xlabel('Количество итераций', fontsize=12)ax.set_ylabel('Learning Rate', fontsize=12)ax.set_title('Тепловая карта Accuracy (Градиентный спуск)', fontsize=14, fontweight='bold')for i in range(len(learning_rates_all)):    for j in range(len(n_iters_all)):        clr = 'white' if heatmap_acc[i, j] > 0.73 else 'black'        ax.text(j, i, f'{heatmap_acc[i, j]:.3f}', ha='center', va='center', fontsize=10, color=clr)plt.colorbar(im, ax=ax, shrink=0.8)plt.tight_layout()plt.show()

## 13. Финальная модель с лучшими параметрами

In [ ]:
best_model = LogisticRegressionGD(learning_rate=0.1, n_iterations=10000)best_model.fit(X_train, y_train)y_pred_best = best_model.predict(X_test)metrics_best = compute_metrics(y_test, y_pred_best)print('Параметры: learning_rate=0.1, n_iterations=10000')print(f'  Accuracy:  {metrics_best["Accuracy"]:.4f}')print(f'  Precision: {metrics_best["Precision"]:.4f}')print(f'  Recall:    {metrics_best["Recall"]:.4f}')print(f'  F1-Score:  {metrics_best["F1-Score"]:.4f}')print(f'  TP={metrics_best["TP"]}, TN={metrics_best["TN"]}, FP={metrics_best["FP"]}, FN={metrics_best["FN"]}')fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(best_model.loss_history, color='#2C3E50', lw=2)axes[0].set_xlabel('Итерация'); axes[0].set_ylabel('Log Loss')axes[0].set_title('Кривая обучения (Loss)', fontweight='bold')axes[0].grid(True, alpha=0.3)axes[0].axhline(y=best_model.loss_history[-1], color='red', ls='--', alpha=0.5,                label=f'Loss: {best_model.loss_history[-1]:.4f}')axes[0].legend()cm = np.array([[metrics_best['TN'], metrics_best['FP']],               [metrics_best['FN'], metrics_best['TP']]])im = axes[1].imshow(cm, cmap='Blues', aspect='auto')axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['Предсказано: 0', 'Предсказано: 1'])axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(['Реально: 0', 'Реально: 1'])axes[1].set_title('Матрица ошибок', fontweight='bold')for i in range(2):    for j in range(2):        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',                     fontsize=16, fontweight='bold',                     color='white' if cm[i, j] > cm.max()/2 else 'black')plt.colorbar(im, ax=axes[1], shrink=0.8)plt.suptitle('Финальная модель логистической регрессии', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print(f'\nВеса признаков:')for name, w in zip(features.columns, best_model.weights.flatten()):    bar = '+' * int(abs(w) * 100) if w > 0 else '-' * int(abs(w) * 100)    print(f'  {name:15s}: {w:+.4f}  {bar}')print(f'  {"bias":15s}: {best_model.bias:+.4f}')

## 14. Выводы

### Влияние learning rate- **lr=0.001**: модель сходится медленно, Accuracy ~0.78.- **lr=0.01–0.1**: оптимальный диапазон, наилучший баланс скорости и точности.- **lr=0.5–1.0**: сходимость стабильна на данном датасете (благодаря нормализации), но на более сложных данных возможна расходимость.- **Рекомендация**: lr ∈ [0.01, 0.1].### Влияние количества итераций- При n_iter=100 модель уже даёт Accuracy ~0.80.- При n_iter=1000+ результаты стабилизируются на плато.- **Рекомендация**: n_iter=5000 для уверенной сходимости.### Сравнение методов оптимизации- **Градиентный спуск**: простая реализация, требует 500–5000 итераций.- **Метод Ньютона**: сходится за 3–5 итераций, но требует вычисления и обращения матрицы Гессе (O(n³)).- Для Titanic оба метода дают идентичные финальные метрики (Accuracy ~0.804).### Итоговые результаты- **Accuracy = 0.8045**, **F1-Score = 0.7244**.- Модель хорошо классифицирует погибших (Recall для класса 0 выше), но несколько хуже выживших.- Логистическая регрессия — эффективный baseline для бинарной классификации.- Качество можно улучшить: feature engineering (Age×Pclass, Fare×Pclass), полиномиальные признаки, регуляризация L2.